# Modelo XGBoost — Fase 6, paso 2: tuning liviano de hiperparámetros

Búsqueda chica de hiperparámetros alrededor de los valores por defecto de XGBoost. Cada combinación se entrena con `X_train` completo (train real, 55 features) y se evalúa en `X_val`. Se entrena sin SMOTE y con `scale_pos_weight = benignos / ataques`, calculado solo con `y_train` y fijo en todas las combinaciones: es la variante C, la que eligió la regla de decisión de `02b_ablacion_smote.ipynb` con el dataset corregido (443,614 filas). Con el dataset original la regla había elegido B, sin pesos; `02_modelo.ipynb` se conserva como ese baseline sin tratamiento del desbalance. El objetivo es ver si algún ajuste de `learning_rate`, `max_depth` o `min_child_weight` mejora al baseline en val. El umbral no se elige acá: todas las métricas por umbral usan 0.5, y la elección del umbral se hace en el paso siguiente.

**Por qué no se usa validación cruzada.** El split es cronológico por bloques (ADR 0001). Partir train en folds aleatorios (`KFold`, `GridSearchCV`, `RandomizedSearchCV`) rompería ese orden: cada fold de validación mezclaría flujos de momentos distintos de la captura con flujos contiguos en el fold de entrenamiento, y sus métricas no medirían lo mismo que val. Por eso cada combinación se entrena una sola vez con todo train y se mide en val.

**Por qué la búsqueda es chica.** El diagnóstico de `docs/auditoria-baseline-xgboost.md`, hecho sobre el baseline anterior (57 features, con SMOTE), mostró que 126 de sus 132 FN (umbral 0.5) son dos grupos de flujos Slowhttptest de las 10:32 muy cortos y rápidos (4 y 31.5 µs entre paquetes en la mediana, contra 10.5 s en los Slowhttptest detectados). Esos flujos no tienen la señal low & slow, y en el mismo rango de probabilidad hay benignos. Cambiar hiperparámetros no le da al modelo una señal que los datos no tienen, así que no se espera una mejora grande. Probar muchas combinaciones contra el mismo val sería ajustar el modelo a ese val en particular: con suficientes intentos, alguna combinación gana por azar.

**Val cumple dos funciones.** Val se usa para el early stopping (decide cuántos árboles usa cada combinación) y para elegir la combinación. Por eso las métricas en val de la combinación elegida son algo optimistas. Test no se carga en este notebook; se reserva para la evaluación final del modelo cerrado.

In [1]:
import sys
import itertools
from pathlib import Path

# Raíz del repo = primera carpeta hacia arriba que contenga src/.
# Así el notebook funciona sin importar desde dónde se abra el kernel.
RAIZ_REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(RAIZ_REPO) not in sys.path:
    sys.path.append(str(RAIZ_REPO))

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

from src.evaluation.metrics import evaluate_model, Timer

DATA_DIR = RAIZ_REPO / "data" / "processed" / "xgboost"
print(f"Raíz del repo: {RAIZ_REPO}")
print(f"Datos:         {DATA_DIR}")
print(f"xgboost {xgb.__version__}")

Raíz del repo: C:\Users\Andy Ortega\Progras\Proyecto de Graduación\low-and-slow-ids
Datos:         C:\Users\Andy Ortega\Progras\Proyecto de Graduación\low-and-slow-ids\data\processed\xgboost
xgboost 3.3.0


### Carga de train y val

Solo train (real, sin SMOTE) y val, igual que en `02_modelo.ipynb`. `y` se lee como array 1D (la columna `label_binaria`).

Se verifica que `X_train.csv` sea el train real (310,529 filas y 8,230 ataques; si trae filas sintéticas, el 01 dejó la versión con SMOTE) y que la matriz tenga 55 features, sin los atajos que quita el pipeline compartido (`Destination Port`, `Source Port`, `Protocol`, `Fwd PSH Flags`) ni las columnas de metadatos del split (`Timestamp`, `bloque`).

`scale_pos_weight` se calcula acá, una sola vez, como benignos / ataques de `y_train` (la misma fórmula que la variante C de `02b_ablacion_smote.ipynb`), y se usa igual en las 18 combinaciones.

In [2]:
X_train = pd.read_csv(DATA_DIR / "X_train.csv")
y_train = pd.read_csv(DATA_DIR / "y_train.csv")["label_binaria"].to_numpy()
X_val = pd.read_csv(DATA_DIR / "X_val.csv")
y_val = pd.read_csv(DATA_DIR / "y_val.csv")["label_binaria"].to_numpy()

assert list(X_train.columns) == list(X_val.columns), "Columnas distintas entre train y val"

# Guarda: X_train.csv debe ser el train real. Si trae filas sintéticas, el 01 dejó la versión con SMOTE.
N_TRAIN_ESPERADO = 310_529
N_ATAQUES_TRAIN_ESPERADO = 8_230
assert len(X_train) == N_TRAIN_ESPERADO, (
    f"Se esperaban {N_TRAIN_ESPERADO:,} filas en el train real, hay {len(X_train):,}. "
    "Revisar que 01_feature_engineering.ipynb guarde el train sin SMOTE."
)
assert y_train.sum() == N_ATAQUES_TRAIN_ESPERADO, (
    f"Se esperaban {N_ATAQUES_TRAIN_ESPERADO:,} ataques en el train real, hay {y_train.sum():,}. "
    "Revisar que 01_feature_engineering.ipynb guarde el train sin SMOTE."
)

N_FEATURES_ESPERADO = 55
assert X_train.shape[1] == N_FEATURES_ESPERADO, f"Se esperaban {N_FEATURES_ESPERADO} features, hay {X_train.shape[1]}"

COLUMNAS_EXCLUIDAS = ["Destination Port", "Source Port", "Protocol", "Fwd PSH Flags", "Timestamp", "bloque"]
presentes = [c for c in COLUMNAS_EXCLUIDAS if c in X_train.columns]
assert not presentes, f"Columnas que no deberían estar en X_train/X_val: {presentes}"

for nombre, X, y in [("train", X_train, y_train), ("val", X_val, y_val)]:
    print(f"{nombre:>5}: X {X.shape} | y {y.shape} | ataques: {y.sum()} ({y.mean():.4%})")

# scale_pos_weight = benignos / ataques, calculado solo con y_train (variante C de 02b).
n_benignos_train = int((y_train == 0).sum())
n_ataques_train = int(y_train.sum())
SCALE_POS_WEIGHT = n_benignos_train / n_ataques_train
print(f"\nscale_pos_weight = {n_benignos_train:,} / {n_ataques_train:,} = {SCALE_POS_WEIGHT:.4f}")

train: X (310529, 55) | y (310529,) | ataques: 8230 (2.6503%)
  val: X (66541, 55) | y (66541,) | ataques: 1884 (2.8313%)

scale_pos_weight = 302,299 / 8,230 = 36.7313


### Dispositivo: GPU (CUDA) si está disponible

Misma verificación que en `02_modelo.ipynb`: un fit de prueba chiquito con `device="cuda"` y, si falla, CPU.

In [3]:
def gpu_disponible_para_xgboost() -> bool:
    if not xgb.build_info().get("USE_CUDA", False):
        return False
    try:
        rng = np.random.default_rng(0)
        X_prueba = rng.normal(size=(200, 5))
        y_prueba = (X_prueba[:, 0] > 0).astype(int)
        xgb.XGBClassifier(n_estimators=2, tree_method="hist", device="cuda").fit(X_prueba, y_prueba)
        return True
    except xgb.core.XGBoostError:
        return False


DEVICE = "cuda" if gpu_disponible_para_xgboost() else "cpu"
print(f"USE_CUDA en el build: {xgb.build_info().get('USE_CUDA')}")
print(f"Dispositivo para entrenar: {DEVICE}")

USE_CUDA en el build: True


Dispositivo para entrenar: cuda


### Grilla de búsqueda

18 combinaciones (3 × 3 × 2):

- `learning_rate`: 0.3, 0.1, 0.05
- `max_depth`: 4, 6, 8
- `min_child_weight`: 1, 5

Todo lo demás queda fijo en las 18 combinaciones (`n_estimators=1000`, `early_stopping_rounds=50`, `eval_metric="aucpr"`, `tree_method="hist"`, `random_state=42`, `n_jobs=-1`, `eval_set=[(X_val, y_val)]`, sin SMOTE y con `scale_pos_weight=SCALE_POS_WEIGHT`, calculado arriba desde `y_train`). Con los learning rates bajos puede que 1000 árboles no alcancen antes de que el early stopping corte; si alguna combinación queda cerca de ese límite, se avisa.

La combinación `learning_rate=0.3`, `max_depth=6`, `min_child_weight=1` son los valores por defecto de XGBoost; con `scale_pos_weight` es el baseline de este notebook y la misma configuración que la variante C de `02b_ablacion_smote.ipynb`. Con los mismos datos e hiperparámetros, esa fila tiene que reproducir la variante C de 02b: best_iteration 52, FP 16 y FN 58.

Por cada combinación se registra `best_iteration`, tiempo de entrenamiento, PR-AUC y ROC-AUC en val, y con umbral 0.5 recall, precision, F1, FP y FN. PR-AUC se calcula con `average_precision_score`, igual que en `02_modelo.ipynb`. No es exactamente el `aucpr` que usa XGBoost para el early stopping (difieren en la cuarta cifra decimal), pero es la misma métrica que se reportó en `02_modelo.ipynb` y `02b_ablacion_smote.ipynb`.

Antes de predecir, cada modelo se pasa a CPU (`set_params(device="cpu")`), por el mismo motivo que en `02_modelo.ipynb`: `X_val` está en memoria de CPU.

In [4]:
PARAMS_FIJOS = {
    "n_estimators": 1000,
    "early_stopping_rounds": 50,
    "eval_metric": "aucpr",
    "tree_method": "hist",
    "random_state": 42,
    "n_jobs": -1,
    "scale_pos_weight": SCALE_POS_WEIGHT,
}

GRILLA = {
    "learning_rate": [0.3, 0.1, 0.05],
    "max_depth": [4, 6, 8],
    "min_child_weight": [1, 5],
}
COMBINACIONES = [dict(zip(GRILLA, valores)) for valores in itertools.product(*GRILLA.values())]
print(f"Combinaciones: {len(COMBINACIONES)}")


def entrenar(combinacion: dict) -> tuple[xgb.XGBClassifier, float]:
    """Entrena con X_train completo y early stopping en val. Devuelve el modelo (ya en cpu) y el tiempo."""
    modelo = xgb.XGBClassifier(**PARAMS_FIJOS, **combinacion, device=DEVICE)
    with Timer() as t:
        modelo.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    modelo.set_params(device="cpu")
    return modelo, t.elapsed


def metricas_val(proba: np.ndarray, umbral: float = 0.5) -> dict:
    pred = (proba >= umbral).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, pred, labels=[0, 1]).ravel()
    return {
        "pr_auc": average_precision_score(y_val, proba),
        "roc_auc": roc_auc_score(y_val, proba),
        "recall": recall_score(y_val, pred, zero_division=0),
        "precision": precision_score(y_val, pred, zero_division=0),
        "f1": f1_score(y_val, pred, zero_division=0),
        "FP": int(fp),
        "FN": int(fn),
    }

Combinaciones: 18


In [5]:
filas = []
for i, combinacion in enumerate(COMBINACIONES, start=1):
    modelo_i, segundos = entrenar(combinacion)
    m = metricas_val(modelo_i.predict_proba(X_val)[:, 1])
    filas.append({
        **combinacion,
        "best_iteration": modelo_i.best_iteration,
        "tiempo_s": segundos,
        **m,
    })
    print(f"[{i:>2}/{len(COMBINACIONES)}] lr={combinacion['learning_rate']:<4} "
          f"depth={combinacion['max_depth']} mcw={combinacion['min_child_weight']} -> "
          f"best_iteration {modelo_i.best_iteration:>4} | PR-AUC {m['pr_auc']:.6f} | "
          f"FP {m['FP']:>3} | FN {m['FN']:>3} | {segundos:.1f}s")

resultados = pd.DataFrame(filas)

sin_corte = resultados[resultados["best_iteration"] >= PARAMS_FIJOS["n_estimators"] - PARAMS_FIJOS["early_stopping_rounds"]]
if not sin_corte.empty:
    print("\nAVISO: estas combinaciones quedaron cerca del límite de n_estimators:")
    print(sin_corte[list(GRILLA)].to_string(index=False))

[ 1/18] lr=0.3  depth=4 mcw=1 -> best_iteration   68 | PR-AUC 0.998685 | FP  13 | FN   3 | 1.1s


[ 2/18] lr=0.3  depth=4 mcw=5 -> best_iteration  138 | PR-AUC 0.998450 | FP  12 | FN  13 | 1.3s


[ 3/18] lr=0.3  depth=6 mcw=1 -> best_iteration   52 | PR-AUC 0.997996 | FP  16 | FN  58 | 1.0s


[ 4/18] lr=0.3  depth=6 mcw=5 -> best_iteration   82 | PR-AUC 0.998612 | FP  14 | FN 137 | 1.2s


[ 5/18] lr=0.3  depth=8 mcw=1 -> best_iteration   41 | PR-AUC 0.998273 | FP  16 | FN  58 | 1.0s


[ 6/18] lr=0.3  depth=8 mcw=5 -> best_iteration   51 | PR-AUC 0.998505 | FP  14 | FN 147 | 1.1s


... (8 outputs left out)

[15/18] lr=0.05 depth=6 mcw=1 -> best_iteration  339 | PR-AUC 0.998230 | FP  13 | FN  58 | 2.2s


[16/18] lr=0.05 depth=6 mcw=5 -> best_iteration  349 | PR-AUC 0.998666 | FP  14 | FN  46 | 2.2s


[17/18] lr=0.05 depth=8 mcw=1 -> best_iteration  294 | PR-AUC 0.998405 | FP  16 | FN  58 | 2.3s


[18/18] lr=0.05 depth=8 mcw=5 -> best_iteration  285 | PR-AUC 0.998356 | FP  13 | FN 152 | 2.6s


### Tabla completa, ordenada por PR-AUC en val

In [6]:
COLUMNAS_TABLA = [
    "learning_rate", "max_depth", "min_child_weight", "best_iteration", "tiempo_s",
    "pr_auc", "roc_auc", "recall", "precision", "f1", "FP", "FN",
]
tabla_ordenada = resultados.sort_values("pr_auc", ascending=False, ignore_index=True)[COLUMNAS_TABLA]
tabla_ordenada.insert(0, "rank", tabla_ordenada.index + 1)

with pd.option_context("display.float_format", "{:.6f}".format, "display.width", 200):
    print(tabla_ordenada.to_string(index=False))

 rank  learning_rate  max_depth  min_child_weight  best_iteration  tiempo_s   pr_auc  roc_auc   recall  precision       f1  FP  FN
    1       0.100000          4                 5             371  2.325093 0.998817 0.999969 0.970276   0.993478 0.981740  12  56
    2       0.300000          4                 1              68  1.054453 0.998685 0.999967 0.998408   0.993136 0.995765  13   3
    3       0.050000          6                 5             349  2.219292 0.998666 0.999964 0.975584   0.992441 0.983940  14  46
    4       0.300000          6                 5              82  1.178805 0.998612 0.999963 0.927282   0.992050 0.958573  14 137
    5       0.050000          4                 5             442  2.762016 0.998612 0.999965 0.978769   0.993534 0.986096  12  40
    6       0.300000          8                 5              51  1.075008 0.998505 0.999959 0.921975   0.992005 0.955708  14 147
    7       0.100000          6                 5             187  1.711858 0.99848

### Verificación: la combinación del baseline reproduce la variante C de `02b_ablacion_smote.ipynb`

La fila `learning_rate=0.3`, `max_depth=6`, `min_child_weight=1` (con `scale_pos_weight`) tiene que dar lo mismo que la variante C de 02b: best_iteration 52, FP 16, FN 58, PR-AUC 0.997996. Si no coincide, se reporta acá y no se sigue como si nada: significaría que algo cambió entre notebooks (datos, versión, dispositivo).

In [7]:
# Variante C de 02b_ablacion_smote.ipynb (dataset corregido, 443,614 filas)
REFERENCIA_BASELINE = {"best_iteration": 52, "FP": 16, "FN": 58, "pr_auc": 0.997996}

fila_baseline = resultados[
    (resultados["learning_rate"] == 0.3)
    & (resultados["max_depth"] == 6)
    & (resultados["min_child_weight"] == 1)
].iloc[0]

diferencias = {
    k: (ref, fila_baseline[k])
    for k, ref in REFERENCIA_BASELINE.items()
    if (round(fila_baseline[k], 6) if k == "pr_auc" else int(fila_baseline[k])) != ref
}
if diferencias:
    print("NO REPRODUCE la variante C de 02b_ablacion_smote.ipynb. Diferencias (esperado, obtenido):")
    for k, (ref, obtenido) in diferencias.items():
        print(f"  {k}: {ref} vs {obtenido}")
else:
    print("Reproduce la variante C de 02b_ablacion_smote.ipynb: "
          f"best_iteration {int(fila_baseline['best_iteration'])} | FP {int(fila_baseline['FP'])} | "
          f"FN {int(fila_baseline['FN'])} | PR-AUC {fila_baseline['pr_auc']:.6f}")

Reproduce la variante C de 02b_ablacion_smote.ipynb: best_iteration 52 | FP 16 | FN 58 | PR-AUC 0.997996


### Regla de selección

Definida antes de ver los resultados de la grilla:

1. **Elegibilidad:** solo cuentan las combinaciones con recall >= 0.75 en val con umbral 0.5.
2. **Criterio principal:** mayor PR-AUC en val.
3. **Margen:** todas las combinaciones elegibles que quedan a menos de 0.0005 del mejor PR-AUC (diferencia estrictamente menor a 0.0005, incluyendo la mejor) se consideran equivalentes. Diferencias de ese tamaño son del orden de lo que cambia entre corridas o entre val y otro conjunto de datos, así que no justifican un modelo más complejo.
4. **Desempate por simplicidad** entre las que quedan dentro del margen: menor `max_depth`; si empatan, menor `best_iteration` (menos árboles). Si todavía empatan, mayor PR-AUC.

El recall se pone como filtro de elegibilidad en vez de verificarlo al final, para que la regla siempre devuelva una combinación que lo cumpla. Con los resultados del baseline (variante C de 02b: recall 0.9692 con umbral 0.5) no se espera que el filtro deje afuera a ninguna.

In [8]:
RECALL_MINIMO = 0.75
MARGEN_PR_AUC = 0.0005

elegibles = resultados[resultados["recall"] >= RECALL_MINIMO]
print(f"Elegibles (recall >= {RECALL_MINIMO}): {len(elegibles)} de {len(resultados)}")
assert not elegibles.empty, f"Ninguna combinación alcanza recall >= {RECALL_MINIMO} con umbral 0.5"

mejor_pr_auc = elegibles["pr_auc"].max()
dentro_margen = (
    elegibles[(mejor_pr_auc - elegibles["pr_auc"]) < MARGEN_PR_AUC]
    .sort_values(["max_depth", "best_iteration", "pr_auc"], ascending=[True, True, False])
)
elegida = dentro_margen.iloc[0]
COMBINACION_ELEGIDA = {
    "learning_rate": float(elegida["learning_rate"]),
    "max_depth": int(elegida["max_depth"]),
    "min_child_weight": int(elegida["min_child_weight"]),
}

print(f"Mejor PR-AUC: {mejor_pr_auc:.6f} | umbral del margen: > {mejor_pr_auc - MARGEN_PR_AUC:.6f}")
print(f"Dentro del margen: {len(dentro_margen)} combinaciones, en orden de desempate:\n")
with pd.option_context("display.float_format", "{:.6f}".format, "display.width", 200):
    print(dentro_margen[COLUMNAS_TABLA].to_string(index=False))
print(f"\nCombinación elegida: {COMBINACION_ELEGIDA}")

Elegibles (recall >= 0.75): 18 de 18
Mejor PR-AUC: 0.998817 | umbral del margen: > 0.998317
Dentro del margen: 12 combinaciones, en orden de desempate:

 learning_rate  max_depth  min_child_weight  best_iteration  tiempo_s   pr_auc  roc_auc   recall  precision       f1  FP  FN
      0.300000          4                 1              68  1.054453 0.998685 0.999967 0.998408   0.993136 0.995765  13   3
      0.300000          4                 5             138  1.293643 0.998450 0.999960 0.993100   0.993627 0.993363  12  13
      0.100000          4                 5             371  2.325093 0.998817 0.999969 0.970276   0.993478 0.981740  12  56
      0.050000          4                 5             442  2.762016 0.998612 0.999965 0.978769   0.993534 0.986096  12  40
      0.050000          4                 1             517  2.966186 0.998468 0.999960 0.952760   0.993359 0.972636  12  89
      0.300000          6                 5              82  1.178805 0.998612 0.999963 0.927282 

### Reentrenamiento y evaluación de la combinación elegida

Se vuelve a entrenar la combinación elegida desde cero, con los mismos parámetros, y se evalúa en val con umbral 0.5 usando `evaluate_model`. `save_dir=None` para que no escriba `reports/metrics/`: el modelo todavía no está cerrado (falta el umbral) y esto es val, no test.

El reentrenamiento tiene que dar el mismo resultado que la fila de la grilla; si no, se avisa.

In [9]:
modelo_elegido, tiempo_elegido = entrenar(COMBINACION_ELEGIDA)
y_proba_val = modelo_elegido.predict_proba(X_val)[:, 1]
y_pred_val = (y_proba_val >= 0.5).astype(int)

resultados_elegido = evaluate_model(
    y_val,
    y_pred_val,
    y_proba_val,
    model_name="xgboost_tuning_val",
    training_time_seconds=tiempo_elegido,
    save_dir=None,
)

m_elegido = metricas_val(y_proba_val)
print(f"\nPR-AUC (average precision) en val: {m_elegido['pr_auc']:.6f}")
print(f"best_iteration: {modelo_elegido.best_iteration}")

coincide = (
    modelo_elegido.best_iteration == elegida["best_iteration"]
    and m_elegido["FP"] == elegida["FP"]
    and m_elegido["FN"] == elegida["FN"]
    and abs(m_elegido["pr_auc"] - elegida["pr_auc"]) < 1e-9
)
print("El reentrenamiento coincide con la fila de la grilla." if coincide
      else "AVISO: el reentrenamiento NO coincide con la fila de la grilla.")


=== Resultados: xgboost_tuning_val ===
Accuracy:  0.9998
Precision: 0.9931
Recall:    0.9984  <- métrica principal
F1-score:  0.9958
ROC-AUC:   1.0000
Matriz de confusión -> TN: 64644, FP: 13, FN: 3, TP: 1881
Tiempo de entrenamiento: 0.77s

PR-AUC (average precision) en val: 0.998685
best_iteration: 68
El reentrenamiento coincide con la fila de la grilla.


### Elegida vs baseline (val, umbral 0.5)

El baseline es la fila `0.3 / 6 / 1` de la grilla, con `scale_pos_weight`, que ya se verificó contra la variante C de `02b_ablacion_smote.ipynb` más arriba.

In [10]:
def resumen_fila(best_iteration, tiempo_s, m):
    return {
        "best_iteration": int(best_iteration),
        "tiempo_s": tiempo_s,
        "pr_auc": m["pr_auc"],
        "roc_auc": m["roc_auc"],
        "recall": m["recall"],
        "precision": m["precision"],
        "f1": m["f1"],
        "FP": int(m["FP"]),
        "FN": int(m["FN"]),
    }


etiqueta_elegida = (f"elegida ({COMBINACION_ELEGIDA['learning_rate']} / "
                    f"{COMBINACION_ELEGIDA['max_depth']} / {COMBINACION_ELEGIDA['min_child_weight']})")
comparacion = pd.DataFrame({
    "baseline (0.3 / 6 / 1)": resumen_fila(fila_baseline["best_iteration"], fila_baseline["tiempo_s"], fila_baseline),
    etiqueta_elegida: resumen_fila(modelo_elegido.best_iteration, tiempo_elegido, m_elegido),
})
comparacion["diferencia"] = comparacion.iloc[:, 1] - comparacion.iloc[:, 0]

with pd.option_context("display.float_format", "{:.6f}".format, "display.width", 200):
    print(comparacion.to_string())

                baseline (0.3 / 6 / 1)  elegida (0.3 / 4 / 1)  diferencia
best_iteration               52.000000              68.000000   16.000000
tiempo_s                      1.034152               0.774361   -0.259792
pr_auc                        0.997996               0.998685    0.000689
roc_auc                       0.999940               0.999967    0.000027
recall                        0.969214               0.998408    0.029193
precision                     0.991314               0.993136    0.001822
f1                            0.980140               0.995765    0.015625
FP                           16.000000              13.000000   -3.000000
FN                           58.000000               3.000000  -55.000000


### Umbral

No se elige umbral en este notebook. Todas las métricas por umbral de arriba usan 0.5 para que las combinaciones sean comparables entre sí y con el baseline. La elección del umbral sobre el modelo elegido se hace en el paso siguiente.

### Resumen

In [11]:
print("=" * 60)
print("RESUMEN — Fase 6, paso 2: tuning liviano XGBoost (solo val)")
print("=" * 60)
print(f"Dispositivo:              {DEVICE}")
print(f"Combinaciones probadas:   {len(resultados)} (sin validación cruzada)")
print(f"scale_pos_weight:         {SCALE_POS_WEIGHT:.4f} (benignos / ataques de y_train, fijo en la grilla)")
print(f"Baseline reproducido:     {'sí' if not diferencias else 'NO, ver verificación'}")
print(f"Mejor PR-AUC en val:      {mejor_pr_auc:.6f}")
print(f"Dentro del margen 0.0005: {len(dentro_margen)} combinaciones")
print(f"Combinación elegida:      learning_rate={COMBINACION_ELEGIDA['learning_rate']} | "
      f"max_depth={COMBINACION_ELEGIDA['max_depth']} | min_child_weight={COMBINACION_ELEGIDA['min_child_weight']}")
print(f"  best_iteration {modelo_elegido.best_iteration} | PR-AUC {m_elegido['pr_auc']:.6f} | "
      f"ROC-AUC {m_elegido['roc_auc']:.6f}")
print(f"  umbral 0.5 -> precision {m_elegido['precision']:.4f} | recall {m_elegido['recall']:.4f} | "
      f"F1 {m_elegido['f1']:.4f} | FP {m_elegido['FP']} | FN {m_elegido['FN']}")
print(f"Baseline (0.3 / 6 / 1):   PR-AUC {fila_baseline['pr_auc']:.6f} | FP {int(fila_baseline['FP'])} | "
      f"FN {int(fila_baseline['FN'])}")

RESUMEN — Fase 6, paso 2: tuning liviano XGBoost (solo val)
Dispositivo:              cuda
Combinaciones probadas:   18 (sin validación cruzada)
scale_pos_weight:         36.7313 (benignos / ataques de y_train, fijo en la grilla)
Baseline reproducido:     sí
Mejor PR-AUC en val:      0.998817
Dentro del margen 0.0005: 12 combinaciones
Combinación elegida:      learning_rate=0.3 | max_depth=4 | min_child_weight=1
  best_iteration 68 | PR-AUC 0.998685 | ROC-AUC 0.999967
  umbral 0.5 -> precision 0.9931 | recall 0.9984 | F1 0.9958 | FP 13 | FN 3
Baseline (0.3 / 6 / 1):   PR-AUC 0.997996 | FP 16 | FN 58
